In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import torchvision.datasets as datasets



img_height = 32
img_width = 32
img_channels = 3
latent_dim = 64
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")




class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super(Encoder, self).__init__()
        self.conv1 = nn.Conv2d(img_channels, 32, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc_mean = nn.Linear(128, latent_dim)
        self.fc_log_var = nn.Linear(128, latent_dim)
        
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        z_mean = self.fc_mean(x)
        z_log_var = self.fc_log_var(x)
        return z_mean, z_log_var


     
    
def sampling(z_mean, z_log_var):
    std = torch.exp(0.5 * z_log_var)
    eps = torch.randn_like(std)
    return z_mean + eps * std



class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super(Decoder, self).__init__()
        self.fc = nn.Linear(latent_dim, 8 * 8 * 64)
        self.deconv1 = nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.deconv2 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.deconv3 = nn.ConvTranspose2d(32, img_channels, kernel_size=3, padding=1)
        
    def forward(self, z):
        x = torch.relu(self.fc(z))
        x = x.view(-1, 64, 8, 8)
        x = torch.relu(self.deconv1(x))
        x = torch.relu(self.deconv2(x))
        x = torch.sigmoid(self.deconv3(x))
        return x

      
    
class VAE(nn.Module):
    def __init__(self, latent_dim):
        super(VAE, self).__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
        
    def forward(self, x):
        z_mean, z_log_var = self.encoder(x)
        z = sampling(z_mean, z_log_var)
        reconstructed = self.decoder(z)
        return reconstructed, z_mean, z_log_var

    def vae_loss(self, reconstructed, x, z_mean, z_log_var):
        recon_loss = nn.functional.mse_loss(reconstructed, x, reduction="sum")
        kl_loss = -0.5 * torch.sum(1 + z_log_var - z_mean.pow(2) - z_log_var.exp())
        return recon_loss + kl_loss

    

epochs = 50
batch_size = 32
learning_rate = 0.001



transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.CIFAR10(
    root='/kaggle/input/datasets/pankrzysiu/cifar10-python',  # updated path
    train=True,
    download=False,  # important for Kaggle
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


vae = VAE(latent_dim).to(device)
optimizer = optim.Adam(vae.parameters(), lr=learning_rate)


vae.train()
for epoch in range(epochs):
    total_loss = 0
    for x, _ in train_loader:  
        x = x.to(device)
        optimizer.zero_grad()
        
        
        reconstructed, z_mean, z_log_var = vae(x)
        
        
        loss = vae.vae_loss(reconstructed, x, z_mean, z_log_var)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(train_loader.dataset):.4f}")

    
     
vae.eval()
with torch.no_grad():
    sample = next(iter(train_loader))[0][:1].to(device)  
    reconstructed_sample, _, _ = vae(sample)
    print("Original Shape:", sample.shape)
    print("Reconstructed Shape:", reconstructed_sample.shape)

In [ ]:
import matplotlib.pyplot as plt

def plot_reconstruction(vae, data_loader):
    vae.eval()
    with torch.no_grad():
        x, _ = next(iter(data_loader))
        x = x.to(device)
        reconstructed, _, _ = vae(x)
        
        
        x = x.cpu()
        reconstructed = reconstructed.cpu()
        
        
        fig, axes = plt.subplots(2, 8, figsize=(16, 4))
        for i in range(8):
            axes[0, i].imshow(x[i].permute(1, 2, 0).squeeze())
            axes[0, i].axis('off')
            axes[1, i].imshow(reconstructed[i].permute(1, 2, 0).squeeze())
            axes[1, i].axis('off')
        plt.show()


        
plot_reconstruction(vae, train_loader)